# Qwen3-VL multi-layout invoice LoRA
This notebook is for model training, **not immediate OCR results**. For OCR without labels or a paid API, use `colab_setup.ipynb` in Accuracy mode. Run this training notebook only after `colab_dataset.ipynb` has pushed at least 80 source-verified full-schema v2 labels to GitHub. Use a fresh GPU runtime. This notebook recreates page images/exports from the public repo, evaluates untouched splits and publishes a passing adapter to a public GitHub Release. It does not require Google Drive. A T4 is experimental; larger contexts may require an A100.

In [ ]:
#@title 1. Configure GitHub-only training (no token needed yet)
WORK_DIR = '/content/invoice_ocr_work'
MODEL_ID = 'Qwen/Qwen3-VL-2B-Instruct' #@param {type:'string'}
EPOCHS = 3 #@param {type:'integer'}
MAX_PIXELS = 602112 #@param {type:'integer'}
MODEL_MAX_LENGTH = 4096 #@param {type:'integer'}
MAX_NEW_TOKENS = 4096 #@param {type:'integer'}
import json, pathlib
WORK = pathlib.Path(WORK_DIR)
print('Temporary training workspace:', WORK, 'Persistent labels/adapter: public GitHub')
print('GITHUB_TOKEN is required only at cell 11, after the held-out gate passes.')

In [ ]:
#@title 2. Clone PDFs/verified labels and download official Qwen trainer
import io, os, pathlib, shutil, subprocess, urllib.request, zipfile
def download_zip(url, destination, user_agent):
    if destination.exists():
        print('Reusing', destination); return
    request = urllib.request.Request(url, headers={'Accept':'application/vnd.github+json','User-Agent':user_agent})
    with urllib.request.urlopen(request, timeout=180) as response:
        body = response.read()
    temp = pathlib.Path(str(destination)+'-download')
    if temp.exists(): raise ValueError(f'Incomplete previous download at {temp}; start a fresh runtime')
    temp.mkdir(parents=True)
    with zipfile.ZipFile(io.BytesIO(body)) as archive:
        archive.extractall(temp)
    shutil.move(str(next(path for path in temp.iterdir() if path.is_dir())), str(destination))
PROJECT_DIR = pathlib.Path('/content/OCR')
QWEN_REPO = pathlib.Path('/content/Qwen3-VL')
if not PROJECT_DIR.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ubaid-148/OCR.git',str(PROJECT_DIR)], check=True)
elif (PROJECT_DIR/'.git').is_dir():
    subprocess.run(['git','-C',str(PROJECT_DIR),'pull','--ff-only'], check=True)
else: raise ValueError('/content/OCR exists but is not a Git clone. Start a fresh runtime.')
public_labels = sorted((PROJECT_DIR/'public_invoice_labels').glob('*.json'))
eligible_labels = 0
for label_path in public_labels:
    label = json.loads(label_path.read_text(encoding='utf-8'))
    if label.get('annotation_version') == 'invoice-ocr-annotation-v2' and label.get('verified') is True and label.get('include_in_training') is True:
        eligible_labels += 1
print(f'GitHub labels ready: {eligible_labels} verified/included v2 of {len(public_labels)} JSON files')
if eligible_labels < 80:
    raise ValueError('Training needs 80 verified labels; use colab_dataset.ipynb first. For OCR without training or paid API, open colab_setup.ipynb in Accuracy mode.')
download_zip('https://api.github.com/repos/QwenLM/Qwen3-VL/zipball/main', QWEN_REPO, 'OCR-Qwen-Training')
QWEN_TRAIN = QWEN_REPO/'qwen-vl-finetune'
import sys
if str(PROJECT_DIR) not in sys.path: sys.path.insert(0, str(PROJECT_DIR))
print(PROJECT_DIR, QWEN_TRAIN)

In [ ]:
#@title 3. Install the official training stack
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.0','accelerate==1.7.0','peft==0.17.1','qwen-vl-utils==0.0.14','pillow','pypdfium2==5.13.0'], check=True)
print('Training packages installed.')

In [ ]:
#@title 4. Verify GitHub labels and GPU before rendering, then rebuild exports
import json, subprocess, sys, torch
from training.invoice_dataset import prepare_workspace, validation_report, export_qwen
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime required.')
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(torch.cuda.get_device_name(0), f'{gpu_gb:.1f} GiB')
if gpu_gb < 14:
    raise RuntimeError('At least a 16 GB-class GPU is required for this profile.')
manifest = prepare_workspace(PROJECT_DIR/'public_invoice_pdfs', WORK, dpi=200, allow_public_pdf_dir=True, labels_dir=PROJECT_DIR/'public_invoice_labels', allow_public_labels=True)
report = validation_report(WORK)
if report['errors']: raise ValueError('Invalid public labels: '+str(report['errors'][:5]))
SUMMARY = export_qwen(WORK, min_verified=80)
print('Verified GitHub dataset:', json.dumps(SUMMARY, indent=2))
subprocess.run([sys.executable,str(PROJECT_DIR/'training'/'patch_qwen_single_gpu.py'),'--qwen-root',str(QWEN_TRAIN)], check=True)
subprocess.run([sys.executable,str(PROJECT_DIR/'training'/'register_qwen_dataset.py'),'--qwen-root',str(QWEN_TRAIN),'--train-json',str(WORK/'exports'/'qwen_train.json'),'--validation-json',str(WORK/'exports'/'qwen_validation.json')], check=True)
ARTIFACTS = WORK/'model_artifacts'
ADAPTER_DIR = ARTIFACTS/'qwen3-vl-2b-invoice-v2'
ARTIFACTS.mkdir(exist_ok=True)

In [ ]:
#@title 5. Reject samples that the trainer would silently truncate
subprocess.run([sys.executable,'-m','training.check_qwen_tokens','--dataset',str(WORK/'exports'/'qwen_train.json'),'--model-id',MODEL_ID,'--max-pixels',str(MAX_PIXELS),'--max-length',str(MODEL_MAX_LENGTH),'--max-new-tokens',str(MAX_NEW_TOKENS)], cwd=PROJECT_DIR, check=True)
print('Every complete train answer fits the configured token budgets.')

In [ ]:
#@title 6. Base-model validation benchmark (not test)
MANIFEST = WORK/'exports'/'evaluation_manifest.jsonl'
BASE_VAL_PRED = ARTIFACTS/'base-validation-predictions.jsonl'
BASE_VAL_METRICS = ARTIFACTS/'base-validation-metrics.json'
subprocess.run([sys.executable,'-m','training.run_vlm_eval','--manifest',str(MANIFEST),'--output',str(BASE_VAL_PRED),'--split','validation','--model-id',MODEL_ID,'--max-pixels',str(MAX_PIXELS),'--max-new-tokens',str(MAX_NEW_TOKENS)], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable,'-m','training.score_predictions','--manifest',str(MANIFEST),'--predictions',str(BASE_VAL_PRED),'--split','validation','--output',str(BASE_VAL_METRICS)], cwd=PROJECT_DIR, check=True)

In [ ]:
#@title 7. Train LoRA on train split only
train_command = [sys.executable,'-m','torch.distributed.run','--nproc_per_node=1','qwenvl/train/train_qwen.py',
 '--model_name_or_path',MODEL_ID,'--dataset_use','private_invoice_train',
 '--tune_mm_vision','False','--tune_mm_mlp','False','--tune_mm_llm','True',
 '--lora_enable','True','--lora_r','16','--lora_alpha','32','--lora_dropout','0.05',
 '--fp16','True','--output_dir',str(ADAPTER_DIR),'--num_train_epochs',str(EPOCHS),
 '--per_device_train_batch_size','1','--gradient_accumulation_steps','8',
 '--learning_rate','5e-6','--weight_decay','0.01','--warmup_ratio','0.05','--lr_scheduler_type','cosine',
 '--model_max_length',str(MODEL_MAX_LENGTH),'--max_pixels',str(MAX_PIXELS),'--min_pixels','12544',
 '--gradient_checkpointing','True','--save_strategy','steps','--save_steps','20','--save_total_limit','2',
 '--logging_steps','1','--dataloader_num_workers','2','--report_to','none','--remove_unused_columns','False']
subprocess.run(train_command, cwd=QWEN_TRAIN, check=True)
print('Adapter saved privately at', ADAPTER_DIR)

In [ ]:
#@title 8. Adapter validation and non-regression gate
ADAPTER_VAL_PRED = ARTIFACTS/'adapter-validation-predictions.jsonl'
ADAPTER_VAL_METRICS = ARTIFACTS/'adapter-validation-metrics.json'
subprocess.run([sys.executable,'-m','training.run_vlm_eval','--manifest',str(MANIFEST),'--output',str(ADAPTER_VAL_PRED),'--split','validation','--model-id',MODEL_ID,'--adapter-dir',str(ADAPTER_DIR),'--max-pixels',str(MAX_PIXELS),'--max-new-tokens',str(MAX_NEW_TOKENS)], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable,'-m','training.score_predictions','--manifest',str(MANIFEST),'--predictions',str(ADAPTER_VAL_PRED),'--split','validation','--output',str(ADAPTER_VAL_METRICS)], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable,'-m','training.compare_metrics','--base',str(BASE_VAL_METRICS),'--candidate',str(ADAPTER_VAL_METRICS),'--min-critical','0.95','--min-exact','0.70'], cwd=PROJECT_DIR, check=True)
print('Validation gate passed. Test split is now unlocked.')

In [ ]:
#@title 9. One-time untouched test benchmark and final gate
BASE_TEST_PRED = ARTIFACTS/'base-test-predictions.jsonl'
ADAPTER_TEST_PRED = ARTIFACTS/'adapter-test-predictions.jsonl'
BASE_TEST_METRICS = ARTIFACTS/'base-test-metrics.json'
ADAPTER_TEST_METRICS = ARTIFACTS/'adapter-test-metrics.json'
for output, adapter in ((BASE_TEST_PRED,None),(ADAPTER_TEST_PRED,ADAPTER_DIR)):
    command = [sys.executable,'-m','training.run_vlm_eval','--manifest',str(MANIFEST),'--output',str(output),'--split','test','--model-id',MODEL_ID,'--max-pixels',str(MAX_PIXELS),'--max-new-tokens',str(MAX_NEW_TOKENS)]
    if adapter: command += ['--adapter-dir',str(adapter)]
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable,'-m','training.score_predictions','--manifest',str(MANIFEST),'--predictions',str(BASE_TEST_PRED),'--split','test','--output',str(BASE_TEST_METRICS)], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable,'-m','training.score_predictions','--manifest',str(MANIFEST),'--predictions',str(ADAPTER_TEST_PRED),'--split','test','--output',str(ADAPTER_TEST_METRICS)], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable,'-m','training.compare_metrics','--base',str(BASE_TEST_METRICS),'--candidate',str(ADAPTER_TEST_METRICS),'--min-critical','0.98','--min-exact','0.90'], cwd=PROJECT_DIR, check=True)
print('FINAL HELD-OUT GATE PASSED. Keep the metrics with the adapter; deployment still requires pipeline integration and review safeguards.')

In [ ]:
#@title 10. Write a private deployment approval only after the final gate
from training.invoice_dataset import PROMPT_VERSION
approval = {'approval_version':'invoice-adapter-approval-v1','approved':True,'prompt_version':PROMPT_VERSION,
            'model_id':MODEL_ID,'adapter_dir':str(ADAPTER_DIR.resolve()),
            'base_test_metrics':str(BASE_TEST_METRICS.resolve()),
            'adapter_test_metrics':str(ADAPTER_TEST_METRICS.resolve()),
            'min_critical':0.98,'min_exact':0.90,
            'max_pixels':MAX_PIXELS,'max_new_tokens':MAX_NEW_TOKENS}
from training.adapter_service import adapter_digest, file_digest, load_approval
approval['adapter_sha256'] = adapter_digest(ADAPTER_DIR)
approval['base_test_sha256'] = file_digest(BASE_TEST_METRICS)
approval['adapter_test_sha256'] = file_digest(ADAPTER_TEST_METRICS)
APPROVAL_PATH = ARTIFACTS/'adapter_approval.json'
APPROVAL_PATH.write_text(json.dumps(approval, indent=2), encoding='utf-8')
load_approval(APPROVAL_PATH)
print('Approved adapter can be selected in colab_setup.ipynb:', APPROVAL_PATH)

In [ ]:
#@title 11. Publish the approved adapter and metrics to a PUBLIC GitHub Release
from google.colab import userdata
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception as error:
    raise RuntimeError('To publish: add GITHUB_TOKEN in Colab Secrets and enable Notebook access. Do not paste the token into a cell.') from error
if not GITHUB_TOKEN:
    raise ValueError('GITHUB_TOKEN is empty in Colab Secrets. Add a token with Contents: Read and write for ubaid-148/OCR.')
from training.github_release import bundle_adapter, publish_bundle
BUNDLE_PATH, RELEASE_TAG = bundle_adapter(APPROVAL_PATH, ARTIFACTS/'invoice-adapter-v2.zip')
release_result = publish_bundle(BUNDLE_PATH, RELEASE_TAG, GITHUB_TOKEN)
print(json.dumps(release_result, indent=2))
print('Copy this TRAINED_RELEASE_TAG into colab_setup.ipynb:', RELEASE_TAG)